# 07. State Machine DP: Stocks and Painting

Sometimes at each step you are in one of a **few states** (holding a stock or not, last color used). The DP tracks the **best value per state** and how states transition.

**Learning order:** IDEA → STOCK I → STOCK II (REF) → COOLDOWN → TRANSACTION FEE → AT MOST K → PAINT HOUSE → INTERVIEW THINKING

## 1. Idea

Draw states as circles and day-by-day actions as arrows. For stocks:

- **hold**: I own a share. `hold = max(hold (keep), cash - price (buy))`
- **cash**: I own nothing. `cash = max(cash (wait), hold + price (sell))`

The answer is the best **cash** state at the end. The "state machine" view makes variants (cooldown, fee, limited transactions) easy: add or change states.

In [1]:
def max_profit_one(prices):                           # at most one transaction
    low, best = float("inf"), 0
    for p in prices:
        low = min(low, p)
        best = max(best, p - low)
    return best

print(max_profit_one([7, 1, 5, 3, 6, 4]), max_profit_one([7, 6, 4, 3, 1]))   # 5 0

5 0


## 2. Unlimited transactions with a **cooldown**

After selling, you must wait one day before buying. Three states:
- `hold`: own a share
- `sold`: just sold today (next day is cooldown)
- `rest`: no share, free to buy

Transitions per price `p`:
- `hold = max(hold, rest - p)`
- `sold = hold_prev + p`
- `rest = max(rest, sold_prev)`

In [2]:
def max_profit_cooldown(prices):
    hold, sold, rest = float("-inf"), 0, 0
    for p in prices:
        prev_sold = sold
        sold = hold + p
        hold = max(hold, rest - p)
        rest = max(rest, prev_sold)
    return max(sold, rest)

print(max_profit_cooldown([1, 2, 3, 0, 2]))   # 3

3


## 3. Unlimited transactions with a **fee**

Pay the fee when you sell (or buy). Two states only: `cash` and `hold`.

In [3]:
def max_profit_fee(prices, fee):
    cash, hold = 0, -prices[0]
    for p in prices[1:]:
        cash = max(cash, hold + p - fee)
        hold = max(hold, cash - p)
    return cash

print(max_profit_fee([1, 3, 2, 8, 4, 9], 2))   # 8

8


## 4. At most **K** transactions

State: `buy[t]` = best profit while holding after `t` transactions, `sell[t]` = best profit with no share after `t` transactions. For each price update `t = 1..k`:
- `buy[t] = max(buy[t], sell[t-1] - p)`
- `sell[t] = max(sell[t], buy[t] + p)`

If `k >= n // 2`, it is the same as unlimited transactions (sum of positive differences). `k = 2` is the classical "Stock III".

In [4]:
def max_profit_k(k, prices):
    n = len(prices)
    if k >= n // 2:
        return sum(max(0, prices[i] - prices[i - 1]) for i in range(1, n))
    buy = [float("-inf")] * (k + 1)
    sell = [0] * (k + 1)
    for p in prices:
        for t in range(1, k + 1):
            buy[t] = max(buy[t], sell[t - 1] - p)
            sell[t] = max(sell[t], buy[t] + p)
    return sell[k]

print(max_profit_k(2, [3, 2, 6, 5, 0, 3]), max_profit_k(2, [2, 4, 1]), max_profit_k(2, [3, 3, 5, 0, 0, 3, 1, 4]))   # 7 2 6

7 2 6


## 5. Paint House (state = last color)

Paint a row of houses with 3 colors; no two adjacent houses share a color; minimize cost. `dp[i][c]` = min cost for houses `0..i` with house `i` painted `c`: `cost[i][c] + min(dp[i-1][other two])`. Only the previous row is needed.

In [5]:
def min_cost_paint(costs):
    prev = costs[0][:]
    for row in costs[1:]:
        prev = [row[c] + min(prev[(c + 1) % 3], prev[(c + 2) % 3]) for c in range(3)]
    return min(prev)

print(min_cost_paint([[17, 2, 17], [16, 16, 5], [14, 3, 19]]))   # 10

10


## 6. Interview thinking

**Signals:** "buy and sell", "cooldown", "at most k transactions", "no two adjacent have the same color/type", "last action matters".

**Say:** "I define states for each situation (holding / not holding / cooldown) and the best profit in each. Each day updates them from the previous day's states. O(n) time, O(1) space (O(k) for k transactions)."

**Common mistakes:** updating states in the wrong order (using today's value instead of yesterday's), wrong initial value of `hold` (`-inf` or `-prices[0]`), forgetting the `k ≥ n/2` shortcut, returning `hold` instead of the cash state.

## Quick Revision

- Stocks = states `hold` / `cash` (+ `sold` for cooldown).
- Fee: subtract on sell. K transactions: arrays `buy[t]`, `sell[t]`.
- Paint house: `dp[i][color]`.
- Use the previous day's values when updating (copy or order the updates carefully).